# ASD-XMF X4 Fusion Report
XMF fusion vs 6 fusion baselines, ablations and shortcut tests, XAI, statistics and figures, app export (NB08-NB12).

Each section is one former notebook, run as its own process. A section waits for the ones it needs; a time guard stops before Kaggle's 12-hour limit and the next run continues where this one stopped.

| section | source | needs |
|---|---|---|
| `nb08` | `nb08_fusion.py` | - |
| `nb09` | `nb09_ablations_shortcuts.py` | nb08 |
| `nb10` | `nb10_xai.py` | nb08 |
| `nb12` | `nb12_export_app.py` | nb08 |
| `nb11` | `nb11_stats_figures.py` | nb09, nb10 |

In [ ]:
RUN_PROFILE = "full"   # "full" on GPU; "preflight" = quick CPU smoke test (tools/push.py)
PREFLIGHT = RUN_PROFILE == "preflight"
import os, sys, glob, json, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _find_lib():
    for c in ("/kaggle/input/asd-xmf-lib", "/kaggle/input/datasets/shazedaoni/asd-xmf-lib"):
        if os.path.exists(os.path.join(c, "asd_utils.py")):
            return c
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + "asd_utils.py")
        if hits:
            return os.path.dirname(hits[0])
    return os.path.abspath(os.path.join(os.getcwd(), "..", "..", "asd_lib"))  # local run

def _require_internet():
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=10).close()
    except OSError:
        raise SystemExit("NO INTERNET in this Kaggle session: verify the account's phone number (Kaggle > Settings) "
                         "and keep 'Internet on'. Stopping so no step runs with missing models or features.")
_require_internet()

LIB = _find_lib()
sys.path.insert(0, LIB)
import asd_utils as U
print("asd_lib:", LIB, "| profile:", RUN_PROFILE)
ENV = U.env_report()
open(os.path.join(str(U.WORK), "_run_profile_" + RUN_PROFILE + ".txt"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))  # queue marker

In [ ]:
import asd_runner as R
R.setup('shazedaoni/asd-xmf-x4-fusion-report', RUN_PROFILE, [{"name": "nb08", "after": [], "rev": 2}, {"name": "nb09", "after": ["nb08"], "rev": 2}, {"name": "nb10", "after": ["nb08"], "rev": 2}, {"name": "nb12", "after": ["nb08"], "rev": 2}, {"name": "nb11", "after": ["nb09", "nb10"], "rev": 2}], hours=11.0, rev=2)

## Section `nb08`: NB08 - BanglaASD-XMF and six fusion baselines

In [ ]:
%%writefile {R.SEC_DIR}/nb08.py
# # NB08 - BanglaASD-XMF and six fusion baselines
# **Proposed (XMF).** Face-video branch (2-layer temporal transformer over NB05 expression
# embeddings + 64 behaviour features per frame), voice, cry and M-CHAT branches -> learned
# *missing* tokens -> bidirectional cross-modal attention (face <-> voice/cry) -> clip-to-child
# attention pooling -> gated fusion -> evidential head (P(ASD) + uncertainty -> "refer").
# Trained with modality dropout (1-2 modalities hidden per child) so it works for preverbal
# children and partial recordings; temperature-scaled on an inner split.
#
# **Baselines** (same bundle, folds and seeds): early concatenation, late averaging,
# stacking, Tensor Fusion Network, Low-rank Multimodal Fusion, MulT-style crossmodal
# transformer.  Every model is run **with and without the questionnaire branch**, because
# M-CHAT overlaps with the screening reference (plan 5.1).
#
# 5 folds x 3 seeds per model; two jobs at a time on the two T4s.  Fold models of XMF are
# saved for NB10 (XAI) and NB12 (export).
# 

import numpy as np, pandas as pd
from pathlib import Path
import asd_models as M
OUT = U.WORK / "nb08"; (OUT / "ckpt").mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
EMB = U.find_path("emb_efficientnet_b0_expr.npy") or U.find_path("emb_imagenet_effb0_pca.npy")  # fallback if NB03 weights are missing
print("face embedding:", EMB)
AUD = U.find_path("voice_fusion_index.csv")
AUD = AUD.parent if AUD else None
man = pd.read_csv(MAN)
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
SEEDS = (0,) if PREFLIGHT else (0, 1, 2)
EPOCHS = 3 if PREFLIGHT else 150
bundle = OUT / "bundle.npz"
B = M.build_fusion_bundle(MAN, EMB, FACE, AUD, out_path=bundle)
print({k: v.shape for k, v in B.items() if isinstance(v, np.ndarray)})
print("modality availability (children):", {m: float(B[f"{m}_mask"].reshape(len(B["y"]), -1).any(1).mean()) for m in M.MODALITIES if m in B})


ALL = ["face", "voice", "cry", "quest"]
NOQ = ["face", "voice", "cry"]
MODELS = {
    "xmf": ("xmf", {"mods": ALL}), "xmf_noquest": ("xmf", {"mods": NOQ}),
    "early_concat": ("early_concat", {"mods": ALL}), "early_concat_noquest": ("early_concat", {"mods": NOQ}),
    "late_avg": ("late_avg", {"mods": ALL}), "late_avg_noquest": ("late_avg", {"mods": NOQ}),
    "stacking": ("stacking", {"mods": ALL}), "stacking_noquest": ("stacking", {"mods": NOQ}),
    "tfn": ("tfn", {"mods": ALL}), "tfn_noquest": ("tfn", {"mods": NOQ}),
    "lmf": ("lmf", {"mods": ALL}), "lmf_noquest": ("lmf", {"mods": NOQ}),
    "mult": ("mult", {"mods": ALL}), "mult_noquest": ("mult", {"mods": NOQ}),
}
if PREFLIGHT:
    MODELS = {k: v for k, v in MODELS.items() if k in ("xmf", "xmf_noquest", "early_concat", "stacking", "tfn", "mult")}
jobs = []
for tag, (arch, cfg) in MODELS.items():
    for s in SEEDS:
        for f in FOLDS:
            jobs.append({"name": f"{tag}_s{s}_f{f}", "model": arch, "model_tag": f"fusion_{tag}", "seed": s, "fold": int(f),
                         "bundle": str(bundle), "manifest": str(MAN), "out_dir": str(OUT),
                         "cfg": {**cfg, "epochs": EPOCHS, "modality_dropout": 0.5, "max_drop": 2},
                         "missing_eval": tag.startswith("xmf"), "save_model": tag.startswith("xmf")})
with U.Timer("fusion"):
    codes = U.run_jobs_parallel("asd_models", "fusion_fold_task", jobs, log_dir=OUT / "logs")
print("non-zero exits:", sum(c != 0 for c in codes))


# ## Quick look (full statistics in NB11)
# 

preds = U.load_all_preds([OUT / "preds"])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    t = s.get("test", {})
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": t.get("auroc"),
                 "test_sens": t.get("sens"), "test_spec": t.get("spec"), "test_ece": t.get("ece")})
res = pd.DataFrame(rows).sort_values("oof_auroc", ascending=False)
print(U.table_md(res))
res.to_csv(OUT / "fusion_summary.csv", index=False)
rob = [json.loads(p.read_text()).get("missing_auc", {}) for p in (OUT / "flags").glob("xmf_s*_f*.done")]
if rob:
    print("XMF test AUROC with modalities removed at test time (mean over folds):")
    print(pd.DataFrame(rob).mean().round(3).to_string())
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb08')

## Section `nb09`: NB09 - Ablations, missing-modality robustness and shortcut tests (plan 5.4)

In [ ]:
%%writefile {R.SEC_DIR}/nb09.py
# # NB09 - Ablations, missing-modality robustness and shortcut tests (plan 5.4)
# **Ablations of XMF** (same folds/seeds): remove each modality; remove the handcrafted
# behaviour features; no modality dropout; no evidential head (plain BCE); no temporal
# transformer (frame mean); no cross-modal attention; clip-level vs child-level decisions.
#
# **Shortcut tests - a model that only sees these must fail:**
# 1. face-masked background thumbnails -> ResNet-18
# 2. recording metadata (device, room, recorder, site) -> logistic regression
# 3. image-quality statistics (brightness, contrast, blur) -> logistic regression
# 4. face-detection rate only
# If any shortcut's AUROC CI lies above 0.6, the data have a confound to fix **before** publishing.
# 

import numpy as np, pandas as pd
from pathlib import Path
import asd_models as M
OUT = U.WORK / "nb09"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
BUNDLE = U.find_path("bundle.npz")
man = pd.read_csv(MAN)
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
SEEDS = (0,) if PREFLIGHT else (0, 1, 2)
EPOCHS = 3 if PREFLIGHT else 150
ALL = ["face", "voice", "cry", "quest"]
ABL = {
    "xmf_minus_face": {"mods": ["voice", "cry", "quest"]}, "xmf_minus_voice": {"mods": ["face", "cry", "quest"]},
    "xmf_minus_cry": {"mods": ["face", "voice", "quest"]}, "xmf_minus_quest": {"mods": ["face", "voice", "cry"]},
    "xmf_no_handcrafted": {"mods": ALL, "no_handcrafted": True}, "xmf_no_moddrop": {"mods": ALL, "modality_dropout": 0.0},
    "xmf_no_evidential": {"mods": ALL, "no_evidential": True}, "xmf_no_temporal": {"mods": ALL, "no_temporal": True},
    "xmf_no_cross": {"mods": ALL, "no_cross": True}, "xmf_clip_level": {"mods": ALL, "clip_level": True},
}
if PREFLIGHT:
    ABL = {k: ABL[k] for k in ("xmf_minus_face", "xmf_no_handcrafted", "xmf_clip_level")}
jobs = [{"name": f"{t}_s{s}_f{f}", "model": "xmf", "model_tag": f"ablation_{t}", "seed": s, "fold": int(f),
         "bundle": str(BUNDLE), "manifest": str(MAN), "out_dir": str(OUT),
         "cfg": {"modality_dropout": 0.5, "max_drop": 2, "epochs": EPOCHS, **cfg}}
        for t, cfg in ABL.items() for s in SEEDS for f in FOLDS]
with U.Timer("ablations"):
    U.run_jobs_parallel("asd_models", "fusion_fold_task", jobs, log_dir=OUT / "logs")


# ## Shortcut 1 - background only (face masked)
# 

bgs = OUT / "bgstore"
for d in ("crops", "valid"):
    (bgs / d).mkdir(parents=True, exist_ok=True)
idx = pd.read_csv(FACE / "face_index.csv")
for cid in idx.child_id.unique():
    src = FACE / "bg" / f"{cid}.npy"
    if src.exists():
        a = np.load(src, mmap_mode="r")
        dst = bgs / "crops" / f"{cid}.npy"
        if not dst.exists():
            try:
                os.symlink(src, dst)
            except OSError:
                np.save(dst, np.asarray(a))
        np.save(bgs / "valid" / f"{cid}.npy", np.ones(a.shape[:2], bool))
idx.to_csv(bgs / "face_index.csv", index=False)
jobs = [{"name": f"bg_resnet18_s0_f{f}", "model": "resnet18", "model_tag": "SHORTCUT_background_cnn", "seed": 0, "fold": int(f),
         "manifest": str(MAN), "face_root": str(bgs), "out_dir": str(OUT), "img_size": 112, "epochs": 1 if PREFLIGHT else 10,
         "patience": 3, "batch": 16 if PREFLIGHT else 128, "lr": 1e-3, "k_per_clip": 4, "frozen_epochs": 0,
         "max_steps": 3 if PREFLIGHT else None} for f in FOLDS]
U.run_jobs_parallel("asd_models", "frame_fold_task", jobs, log_dir=OUT / "logs")


# ## Shortcuts 2-4 - metadata, image quality, detection rate (logistic regression, same folds)
# 

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
qual = []
for cid in man.child_id:
    p = FACE / "qual" / f"{cid}.npy"
    v = FACE / "valid" / f"{cid}.npy"
    if p.exists():
        q = np.load(p).astype(np.float32)
        nz = q.sum(-1) != 0
        qq = q[nz] if nz.any() else q.reshape(-1, q.shape[-1])
        qual.append({"child_id": cid, "bright": qq[:, 0].mean(), "contrast": qq[:, 1].mean(), "blur": qq[:, 2].mean(),
                     "det_rate": float(np.load(v).mean())})
qual = pd.DataFrame(qual)
meta_cols = [c for c in ("device", "room", "recorder", "site") if c in man and man[c].nunique() > 1]
X_meta = pd.get_dummies(man.set_index("child_id")[meta_cols].astype(str)) if meta_cols else None
SHORT = {"SHORTCUT_metadata": X_meta,
         "SHORTCUT_image_quality": qual.set_index("child_id")[["bright", "contrast", "blur"]] if len(qual) else None,
         "SHORTCUT_detection_rate": qual.set_index("child_id")[["det_rate"]] if len(qual) else None}
for name, X in SHORT.items():
    if X is None or X.shape[1] == 0:
        print(name, "- not testable (no variation)")
        continue
    for s in SEEDS:
        for f in FOLDS:
            tr, va, te = U.fold_ids(man, s, f)
            tr, va, te = [[c for c in ids if c in X.index] for ids in (tr, va, te)]
            y = man.set_index("child_id").label
            if len(set(y.loc[tr])) < 2:
                continue
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X.loc[tr].astype(float), y.loc[tr])
            for part, ids in (("val", va), ("test", te)):
                if ids:
                    d = pd.DataFrame({"child_id": ids, "prob": clf.predict_proba(X.loc[ids].astype(float))[:, 1], "part": part})
                    d["y_true"] = d.child_id.map(y)
                    U.save_preds(d, name, f, OUT / "preds", s)


# ## Tables
# 

preds = U.load_all_preds([OUT / "preds"])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 2000)
    ci = s["oof_ci"].set_index("metric").loc["auroc"] if "oof_ci" in s else None
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"),
                 "lo": None if ci is None else ci.lo, "hi": None if ci is None else ci.hi,
                 "test_auroc": s.get("test", {}).get("auroc"),
                 "verdict": ("CONFOUND - fix data" if (m.startswith("SHORTCUT") and ci is not None and ci.lo > 0.6) else
                             ("ok (near chance)" if m.startswith("SHORTCUT") else ""))})
res = pd.DataFrame(rows)
print(U.table_md(res))
res.to_csv(OUT / "ablation_shortcut_table.csv", index=False)
clip = [json.loads(p.read_text()) for p in (OUT / "flags").glob("xmf_clip_level_*.done")]
if clip:
    print("clip-level AUROC (mean over folds):", pd.DataFrame(clip).reindex(columns=["clip_auc_val", "clip_auc_test"]).mean().round(3).to_dict())
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb09')

## Section `nb10`: NB10 - Explainable AI at five levels, with numbers (plan 6)

In [ ]:
%%writefile {R.SEC_DIR}/nb10.py
# # NB10 - Explainable AI at five levels, with numbers (plan 6)
# | level | question | method | check |
# |---|---|---|---|
# | modality | did face, voice or cry decide? | exact Shapley over modalities (16 subsets) | sums to P - P(empty) |
# | time | which seconds mattered? | XMF frame attention + Integrated Gradients | rank agreement attention vs IG |
# | behaviour | what did the child do? | SHAP on behaviour features; TCAV concepts (smile, gaze-away) | concept vs random-CAV t-test |
# | pixel | which face regions? | Grad-CAM, Grad-CAM++, Score-CAM, Eigen-CAM | eyes/mouth energy, deletion/insertion AUC, randomisation sanity, stability |
# | case / counterfactual | similar children? what would change it? | kNN in fusion space; single-feature counterfactual | neighbour agreement |
#
# Finally a **blinded clinician-rating packet** (50 explanation cards + rating sheet) for the
# 3-5 clinician study.
# 

import numpy as np, pandas as pd, torch
from pathlib import Path
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import asd_models as M, asd_xai as X
U.pip_install("grad-cam", "shap")
OUT = U.WORK / "nb10"; OUT.mkdir(parents=True, exist_ok=True)
dev = M.device()
man = pd.read_csv(U.find_path("manifest.csv"))
FACE = U.find_path("face_index.csv").parent
store = M.FaceStore(FACE)
B = M.load_bundle(U.find_path("bundle.npz"))
names = json.loads((FACE / "hand_feature_names.json").read_text())
De, Dh = B["face_dims"]
ckpts = sorted(f for d in U.find_all("ckpt") for f in d.glob("xmf_s0_f*.pt"))
print("XMF fold checkpoints:", [p.name for p in ckpts])
SYN = bool(man.is_synthetic.iloc[0])
pos = {c: i for i, c in enumerate(B["child_id"])}


# ## 1. Modality Shapley, frame attention, IG, fusion embedding (per fold model, its own val children)
# 

recs, zs = [], []
for ck in ckpts:
    model, info = M.load_fusion_ckpt(ck, dev)
    Bn = M.apply_norm(B, info["norm"])
    tr, va, te = U.fold_ids(man, 0, info["fold"])
    for part, ids in (("val", va), ("test", te), ("train", info["fit_children"])):
        idx = np.array([pos[c] for c in ids if c in pos])
        for s in range(0, len(idx), 32):
            ii = idx[s:s + 32]
            b = M.batch_from_bundle(Bn, ii, dev, model.mods)
            with torch.no_grad():
                o = model(b, return_all=True)
            zs.append(pd.DataFrame(o["z"].cpu().numpy(), index=B["child_id"][ii]).assign(part=part, fold=info["fold"]))
            if part == "train":
                continue
            phi = M.modality_shapley(model, b, model.mods)
            ig = X.integrated_gradients_frames(model, b, steps=8 if PREFLIGHT else 32) if "face" in model.mods else None
            fa = o.get("frame_attn")
            for k, i in enumerate(ii):
                r = {"child_id": B["child_id"][i], "y": int(B["y"][i]), "part": part, "fold": info["fold"],
                     "prob": float(o["prob"][k]), "unc": float(o["unc"][k]),
                     **{f"shap_{m}": float(phi[k, j]) for j, m in enumerate(model.mods)},
                     **{f"gate_{m}": float(o["gates"][k, j]) for j, m in enumerate(model.mods)}}
                if ig is not None and fa is not None:
                    fm = B["face_frame_mask"][i]
                    a, g = fa[k].cpu().numpy()[fm], np.abs(ig[k])[fm]
                    r["attn_vs_ig_spearman"] = float(pd.Series(a).corr(pd.Series(g), method="spearman")) if len(a) > 3 else np.nan
                    r["ig_frames"] = json.dumps(np.round(np.abs(ig[k]) * fm, 5).tolist())
                recs.append(r)
xai = pd.DataFrame(recs)
xai.drop(columns=["ig_frames"], errors="ignore").to_csv(OUT / "xai_per_child.csv", index=False)
mods = [c[5:] for c in xai.columns if c.startswith("shap_")]
glob_imp = xai[[f"shap_{m}" for m in mods]].abs().mean().rename(lambda c: c[5:])
print("mean |Shapley| per modality:\n", glob_imp.round(4).to_string())
if "attn_vs_ig_spearman" in xai:
    print("attention-vs-IG rank agreement:", round(float(xai.attn_vs_ig_spearman.mean()), 3))


# ## 2. Time: which seconds of each stimulus mattered (IG averaged by stimulus)
# 

fidx = pd.read_csv(FACE / "face_index.csv")
prof = {}
for r in xai.dropna(subset=["ig_frames"]).itertuples():
    ig = np.array(json.loads(r.ig_frames))
    for c, row in fidx[fidx.child_id == r.child_id].iterrows():
        if row.clip_idx < ig.shape[0]:
            prof.setdefault(row.stimulus, []).append(ig[row.clip_idx])
fig, ax = plt.subplots(figsize=(9, 4))
for stim, arr in prof.items():
    a = np.mean(arr, 0)
    ax.plot(np.arange(len(a)) / 5.0, a / (a.max() + 1e-9), label=stim)
ax.set(xlabel="seconds into clip", ylabel="relative |IG|", title="Temporal importance by stimulus")
ax.legend(fontsize=7); plt.tight_layout(); plt.savefig(OUT / "temporal_importance.png", dpi=200); plt.show()


# ## 3. Behaviour: interpretable child-level features + SHAP + counterfactuals
# 

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
KEY = [n for n in names if any(k in n.lower() for k in ("smile", "blink", "browinnerup", "jawopen", "yaw", "pitch", "gaze", "ear_"))]
beh = []
for cid in man.child_id:
    h = store.hand(cid)
    if h is None:
        continue
    st = fidx[fidx.child_id == cid].set_index("clip_idx").stimulus
    row = {"child_id": cid}
    for stim in sorted(fidx.stimulus.unique()):
        cl = [c for c, s_ in st.items() if s_ == stim and c < h.shape[0]]
        if not cl:
            continue
        v = h[cl].reshape(-1, h.shape[-1])
        for n in KEY:
            j = names.index(n)
            val = v[:, j]
            if n in ("yaw", "pitch"):
                val = np.abs(val)
            row[f"{stim}__{n}"] = np.nanmean(val) if np.isfinite(val).any() else np.nan
    beh.append(row)
beh = pd.DataFrame(beh).set_index("child_id")
beh = beh.loc[:, beh.notna().mean() > 0.5]
bcols = beh.columns.tolist()
y = man.set_index("child_id").label
oof = []
for s in (0,) if PREFLIGHT else (0, 1, 2):
    for f in sorted(man[f"fold_s{s}"][man[f"fold_s{s}"] >= 0].unique()):
        tr, va, te = U.fold_ids(man, s, f)
        tr, va, te = [[c for c in ids if c in beh.index] for ids in (tr, va, te)]
        if len(set(y.loc[tr])) < 2:
            continue
        clf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.05, max_iter=5000)).fit(beh.loc[tr], y.loc[tr])
        for part, ids in (("val", va), ("test", te)):
            if ids:
                d = pd.DataFrame({"child_id": ids, "prob": clf.predict_proba(beh.loc[ids])[:, 1], "part": part, "y_true": y.loc[ids].values})
                U.save_preds(d, "behaviour_logreg", f, OUT / "preds", s)
cv_ids = [c for c in man.child_id[man.split == "cv"] if c in beh.index]
clf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.05, max_iter=5000)).fit(beh.loc[cv_ids], y.loc[cv_ids])
Z = clf[:-1].transform(beh)
try:
    import shap
    sv = shap.LinearExplainer(clf[-1], clf[:-1].transform(beh.loc[cv_ids])).shap_values(Z)
    imp = pd.DataFrame({"feature": bcols, "mean_abs_shap": np.abs(sv).mean(0), "coef": clf[-1].coef_[0]}).sort_values("mean_abs_shap", ascending=False)
    print(U.table_md(imp.head(15)))
    imp.to_csv(OUT / "behaviour_shap.csv", index=False)
except Exception as e:
    print("SHAP failed:", e)
thr = 0.5
cfs = []
for i, cid in enumerate(beh.index):
    if cid in set(man.child_id[man.split == "test"]):
        feat, delta = X.single_feature_counterfactual(clf[-1], Z[i], bcols, thr)
        if feat:
            cfs.append({"child_id": cid, "feature": feat, "delta_sd": round(float(delta), 2)})
cfs = pd.DataFrame(cfs)
print("counterfactuals found for", len(cfs), "test children; most common features:")
print(cfs.feature.value_counts().head(8).to_string() if len(cfs) else "-")
cfs.to_csv(OUT / "counterfactuals.csv", index=False)


# ## 4. TCAV-style concepts in the embedding space (smile, gaze-away) vs random directions
# 

from scipy import stats
def frames_where(pred, k=3000):
    E, rows = [], 0
    for i, cid in enumerate(B["child_id"]):
        h = store.hand(cid)
        if h is None:
            continue
        C = min(h.shape[0], B["face"].shape[1])
        m = pred(h[:C]) & B["face_frame_mask"][i, :C]
        if m.any():
            E.append(B["face"][i, :C][m][:, :De].astype(np.float32))
    E = np.concatenate(E) if E else np.zeros((0, De), np.float32)
    return E[np.random.default_rng(0).choice(len(E), min(k, len(E)), replace=False)] if len(E) else E
smile_j = [names.index(n) for n in names if "smile" in n.lower()]
yaw_j = names.index("yaw") if "yaw" in names else None
concepts = {}
if smile_j:
    concepts["social_smile"] = (frames_where(lambda h: np.nanmean(h[..., smile_j], -1) > 0.4), frames_where(lambda h: np.nanmean(h[..., smile_j], -1) < 0.1))
if yaw_j is not None:
    concepts["gaze_away"] = (frames_where(lambda h: np.abs(h[..., yaw_j]) > 20), frames_where(lambda h: np.abs(h[..., yaw_j]) < 8))
tcav_rows = []
if ckpts and concepts:
    model, info = M.load_fusion_ckpt(ckpts[0], dev)
    mu, sd = info["norm"]["face"]
    Bn = M.apply_norm(B, info["norm"])
    te = np.array([pos[c] for c in man.child_id[man.split == "test"] if c in pos])
    b = M.batch_from_bundle(Bn, te, dev, model.mods)
    rng = np.random.default_rng(0)
    for cname, (Ep, En) in concepts.items():
        if len(Ep) < 20 or len(En) < 20:
            print(cname, "too few frames"); continue
        z = lambda E: (E - mu[:De]) / sd[:De]
        s_con = X.tcav_score(model, b, X.concept_vector(z(Ep), z(En)), De)
        rand = [(X.tcav_score(model, b, X.concept_vector(z(En[rng.permutation(len(En))[:len(En)//2]]), z(En[rng.permutation(len(En))[:len(En)//2]]), seed=k), De) > 0).mean() for k in range(10)]
        score = float((s_con > 0).mean())
        t = stats.ttest_1samp(rand, score)
        tcav_rows.append({"concept": cname, "tcav_score": score, "random_mean": float(np.mean(rand)), "p_vs_random": float(t.pvalue)})
tcav = pd.DataFrame(tcav_rows)
print(U.table_md(tcav) if len(tcav) else "TCAV skipped")
tcav.to_csv(OUT / "tcav.csv", index=False)


# ## 5. Pixel level: CAM family on the expression-pretrained frame model (NB04, seed 0)
# 

import timm
frame_ck = sorted(f for d in U.find_all("ckpt") for f in d.glob("efficientnet_b0_expr_s0_f*.pt"))
pix_rows, sanity_rows, stab_rows = [], [], []
if frame_ck:
    fm_ = timm.create_model(M.TIMM_ZOO["efficientnet_b0"], pretrained=False, num_classes=1)
    fm_.load_state_dict(torch.load(frame_ck[0], map_location="cpu"))
    fm_ = fm_.to(dev).eval()
    fold0 = int(frame_ck[0].stem.split("_f")[-1])
    _, va, _ = U.fold_ids(man, 0, fold0)
    imgs, bxs = [], []
    pool_ids = [c for c in va if store.has(c)] or [c for c in man.child_id if store.has(c)]  # fold-0 val children first
    for cid in pool_ids[: 4 if PREFLIGHT else 32]:
        if not store.has(cid):
            continue
        v = store.valid(cid); cr = store.crops(cid)
        bx = np.load(FACE / "boxes" / f"{cid}.npy").astype(np.float32)
        for c in range(min(2, v.shape[0])):
            ts = np.flatnonzero(v[c])
            if len(ts):
                t = ts[len(ts) // 2]; imgs.append(np.asarray(cr[c, t])); bxs.append(bx[c, t])
if frame_ck and imgs:
    x = M.gpu_prepare(torch.from_numpy(np.stack(imgs)).to(dev), 160, train=False).float().contiguous()
    bxs = np.stack(bxs)
    for meth in ("gradcam", "gradcam++", "eigencam") + (() if PREFLIGHT else ("scorecam",)):
        cams = X.compute_cams(fm_, x, meth)
        pix_rows.append({"method": meth, **X.region_energy(cams, bxs), **X.deletion_insertion(fm_, x, cams, steps=10 if PREFLIGHT else 20)})
    rnd = np.random.default_rng(0).random((x.shape[0], 160, 160)).astype(np.float32)
    pix_rows.append({"method": "random_map", **X.region_energy(rnd, bxs), **X.deletion_insertion(fm_, x, rnd, steps=10 if PREFLIGHT else 20)})
    sanity_rows = X.sanity_cascade(fm_, x, "gradcam")
    stab_rows = [X.stability(fm_, x, "gradcam")]
    cams = X.compute_cams(fm_, x[:8], "gradcam")
    fig, axs = plt.subplots(2, min(8, len(cams)), figsize=(2 * min(8, len(cams)), 4))
    for k in range(min(8, len(cams))):
        axs[0, k].imshow(imgs[k]); axs[1, k].imshow(imgs[k]); axs[1, k].imshow(cams[k], alpha=0.5, cmap="jet")
        axs[0, k].axis("off"); axs[1, k].axis("off")
    plt.tight_layout(); plt.savefig(OUT / "gradcam_examples.png", dpi=150); plt.show()
pix = pd.DataFrame(pix_rows); print(U.table_md(pix) if len(pix) else "no frame checkpoint")
print("sanity (should fall towards 0):", sanity_rows); print("stability (should stay high):", stab_rows)
pix.to_csv(OUT / "pixel_xai_metrics.csv", index=False)
U.save_json({"sanity": sanity_rows, "stability": stab_rows}, OUT / "pixel_xai_sanity.json")


# ## 6. Case-based: nearest training children in the fusion embedding
# 

Zs = pd.concat(zs) if zs else pd.DataFrame()
knn = []
if len(Zs):
    from sklearn.neighbors import NearestNeighbors
    for f, g in Zs.groupby("fold"):
        trn, qry = g[g.part == "train"], g[g.part == "test"]
        if len(trn) < 6 or not len(qry):
            continue
        nn_ = NearestNeighbors(n_neighbors=5, metric="cosine").fit(trn.drop(columns=["part", "fold"]).values)
        _, ind = nn_.kneighbors(qry.drop(columns=["part", "fold"]).values)
        for cid, nb in zip(qry.index, ind):
            labs = y.loc[trn.index[nb]].values
            knn.append({"child_id": cid, "fold": f, "neighbour_asd_frac": labs.mean(), "y": y.loc[cid]})
knn = pd.DataFrame(knn)
if len(knn):
    print("kNN neighbour-label AUROC (case-based plausibility):", round(U._safe_auc(knn.y, knn.neighbour_asd_frac), 3))
knn.to_csv(OUT / "case_based_knn.csv", index=False)


# ## 7. Blinded clinician-rating packet (50 cards)
# 

cards = OUT / "clinician_packet"; cards.mkdir(exist_ok=True)
sel = xai[xai.part == "test"].drop_duplicates("child_id").sample(frac=1, random_state=1).head(50)
sheet = []
for k, r in enumerate(sel.itertuples()):
    card_id = f"CARD{k+1:03d}"
    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].barh(mods, [getattr(r, f"shap_{m}") for m in mods]); ax[0].set_title("contribution to risk")
    if isinstance(getattr(r, "ig_frames", None), str):
        ig = np.array(json.loads(r.ig_frames)); ax[1].imshow(ig, aspect="auto", cmap="magma"); ax[1].set(title="when it mattered", xlabel="frame (5 fps)", ylabel="clip")
    fig.suptitle(f"{card_id}  risk={r.prob:.2f}  uncertainty={r.unc:.2f}"); plt.tight_layout()
    plt.savefig(cards / f"{card_id}.png", dpi=110); plt.close()
    sheet.append({"card_id": card_id, "usefulness_1to5": "", "agreement_1to5": "", "comment": ""})
pd.DataFrame(sheet).to_csv(cards / "rating_sheet_BLINDED.csv", index=False)
sel.assign(card_id=[s["card_id"] for s in sheet])[["card_id", "child_id", "y", "prob"]].to_csv(OUT / "rating_key_DO_NOT_SHARE.csv", index=False)
print("cards:", len(sheet), "->", cards)
if SYN:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb10')

## Section `nb12`: NB12 - Export for the offline clinician app

In [ ]:
%%writefile {R.SEC_DIR}/nb12.py
# # NB12 - Export for the offline clinician app
# * frame encoder (NB03 EfficientNet-B0 expression net, 256-d) and the XMF fusion model ->
#   ONNX (opset 17) -> dynamic **INT8** quantisation (ONNX Runtime)
# * numerical parity FP32 vs INT8, parameter count, FLOPs, file size, CPU latency
#   (Kaggle CPU is only a proxy - re-measure on a mid-range Android phone with ONNX Runtime
#   Mobile; MediaPipe runs face landmarks on-device)
# * optional TFLite via onnx2tf (skipped gracefully if it does not install)
# 

import numpy as np, pandas as pd, torch, time
from pathlib import Path
import asd_models as M
U.pip_install("onnx", "onnxruntime", "onnxscript", "onnxconverter-common")
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, quantize_static, QuantType, QuantFormat, CalibrationDataReader
OUT = U.WORK / "export"; OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(4)
report = {}

def flops(model, *args):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc:
            model(*args)
        return int(fc.get_total_flops())
    except Exception as e:
        return None

def onnx_export(model, args, path, **kw):
    """Legacy TorchScript exporter (robust for dict-input models); falls back on older torch."""
    try:
        torch.onnx.export(model, args, path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, args, path, **kw)

def bench(path, feeds, n=30):
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    for _ in range(3): s.run(None, feeds)
    t = time.perf_counter()
    for _ in range(n): out = s.run(None, feeds)
    return (time.perf_counter() - t) / n * 1000, out


# ## 1. Frame encoder (one face crop -> 256-d expression embedding + 7 expression logits)
# 

ck = U.find_path("efficientnet_b0_expr_backbone.pt")
if ck:
    net = M.load_expr_net("efficientnet_b0", ck.parent).eval()
    class Enc(torch.nn.Module):
        def __init__(s, n): super().__init__(); s.n = n
        def forward(s, x): e = s.n.embed(x); return e, s.n.cls_head(e)
    enc = Enc(net).eval()
    x = torch.randn(1, 3, 160, 160)
    p32, p8 = OUT / "frame_encoder_fp32.onnx", OUT / "frame_encoder_int8.onnx"
    onnx_export(enc, x, p32, input_names=["image"], output_names=["embedding", "expr_logits"], opset_version=17,
                      dynamic_axes={"image": {0: "batch"}, "embedding": {0: "batch"}, "expr_logits": {0: "batch"}})
    # CNNs need *static* INT8 (activations calibrated on real face crops); dynamic INT8 only suits Linear layers
    calib = []
    fi = U.find_path("face_index.csv")
    if fi is not None:
        st = M.FaceStore(fi.parent)
        for cid in st.index.child_id.unique()[:16]:
            v = st.valid(cid); cr = st.crops(cid)
            for c in range(v.shape[0]):
                for t in np.flatnonzero(v[c])[::8][:2]:
                    calib.append(M.gpu_prepare(torch.from_numpy(np.asarray(cr[c, t]))[None], 160, train=False).float().numpy())
    class Reader(CalibrationDataReader):
        def __init__(s, xs): s.it = iter({"image": a} for a in xs)
        def get_next(s): return next(s.it, None)
    cal, held = calib[: max(1, len(calib) * 2 // 3)], calib[len(calib) * 2 // 3:]
    def fidelity(path):
        a, b = ort.InferenceSession(str(p32)), ort.InferenceSession(str(path))
        cs = [float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9))
              for u, v in ((a.run(None, {"image": h})[0][0], b.run(None, {"image": h})[0][0]) for h in held)]
        return float(np.mean(cs)) if cs else float("nan")
    chosen, cos8 = p32, float("nan")
    if len(calib) >= 8:
        quantize_static(str(p32), str(p8), Reader(cal), quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
        cos8 = fidelity(p8)
        chosen = p8
    if not (cos8 >= 0.98):  # EfficientNet (SiLU/SE/depthwise) usually fails INT8 -> ship FP16 instead
        import onnx
        from onnxconverter_common import float16
        p16 = OUT / "frame_encoder_fp16.onnx"
        onnx.save(float16.convert_float_to_float16(onnx.load(str(p32)), keep_io_types=True, op_block_list=["Gemm"]), str(p16))
        chosen = p16
    print(f"INT8 held-out embedding cosine = {cos8:.3f} -> shipping {chosen.name}")
    xr = calib[-1] if calib else x.numpy()
    ms32, o32 = bench(p32, {"image": xr}); ms8, o8 = bench(chosen, {"image": xr})
    report["frame_encoder"] = {"params": M.count_params(enc), "flops": flops(enc, x),
                               "mb_fp32": p32.stat().st_size / 1e6, "shipped": chosen.name, "mb_shipped": chosen.stat().st_size / 1e6,
                               "ms_fp32_cpu": ms32, "ms_shipped_cpu": ms8, "calibration_frames": len(calib), "int8_heldout_cosine": cos8,
                               "emb_cosine_fp32_vs_shipped": float(np.dot(o32[0][0], o8[0][0]) / (np.linalg.norm(o32[0][0]) * np.linalg.norm(o8[0][0])))}
    print(report["frame_encoder"])


# ## 2. XMF fusion (fixed shapes: 10 clips x 40 frames; masks mark missing modalities)
# 

cks = sorted(f for d in U.find_all("ckpt") for f in d.glob("xmf_noquest_s0_f*.pt")) or sorted(f for d in U.find_all("ckpt") for f in d.glob("xmf_s0_f*.pt"))
if cks:
    model, info = M.load_fusion_ckpt(cks[0], torch.device("cpu"))
    mods = model.mods
    C, T = 10, 40
    dims = info["dims"]
    ex = {"face": torch.randn(1, C, T, dims["face"]), "face_frame_mask": torch.ones(1, C, T, dtype=torch.bool),
          "face_mask": torch.ones(1, C, dtype=torch.bool)}
    for m in ("voice", "cry", "quest"):
        if m in mods:
            L = {"voice": C, "cry": 3}.get(m)
            ex[m] = torch.randn(1, L, dims[m]) if L else torch.randn(1, dims[m])
            ex[f"{m}_mask"] = torch.ones(1, L, dtype=torch.bool) if L else torch.ones(1, dtype=torch.bool)
    keys = list(ex.keys())
    class Wrap(torch.nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, *args):
            o = s.m(dict(zip(keys, args)))
            return o["prob"], o["unc"], o["gates"]
    w = Wrap(model).eval()
    args = tuple(ex[k] for k in keys)
    f32, f8 = OUT / "xmf_fusion_fp32.onnx", OUT / "xmf_fusion_int8.onnx"
    onnx_export(w, args, f32, input_names=keys, output_names=["p_asd", "uncertainty", "modality_gates"], opset_version=17)
    quantize_dynamic(str(f32), str(f8), weight_type=QuantType.QInt8)
    feeds = {k: v.numpy() for k, v in ex.items()}
    ms32, o32 = bench(f32, feeds); ms8, o8 = bench(f8, feeds)
    with torch.no_grad():
        pt = w(*args)[0].item()
    report["xmf_fusion"] = {"checkpoint": cks[0].name, "modalities": mods, "params": M.count_params(model), "flops": flops(w, *args),
                            "mb_fp32": f32.stat().st_size / 1e6, "mb_int8": f8.stat().st_size / 1e6,
                            "ms_fp32_cpu": ms32, "ms_int8_cpu": ms8, "p_torch": pt, "p_onnx_fp32": float(o32[0][0]), "p_onnx_int8": float(o8[0][0]),
                            "temperature": info["T"], "norm_stats_file": "xmf_norm.json"}
    U.save_json({m: {"mean": mu.tolist(), "sd": sd.tolist()} for m, (mu, sd) in info["norm"].items()} | {"temperature": info["T"], "inputs": keys},
                OUT / "xmf_norm.json")
    print(report["xmf_fusion"])


# ## 3. Optional TFLite
# 

if not PREFLIGHT and (OUT / "frame_encoder_fp32.onnx").exists():
    try:
        if U.pip_install("onnx2tf", "tensorflow", "onnx_graphsurgeon", "sng4onnx"):
            import subprocess
            r = subprocess.run(["onnx2tf", "-i", str(OUT / "frame_encoder_fp32.onnx"), "-o", str(OUT / "tflite"), "-oiqt"], capture_output=True, text=True, timeout=1800)
            print(r.stdout[-1500:], r.stderr[-1500:])
            report["tflite"] = sorted(p.name for p in (OUT / "tflite").glob("*.tflite"))
    except Exception as e:
        report["tflite"] = f"skipped: {e}"
U.save_json(report, OUT / "export_report.json")
print(json.dumps(report, indent=1, default=str))


In [ ]:
R.run('nb12')

## Section `nb11`: NB11 - Statistics, tables and paper figures (CPU only)

In [ ]:
%%writefile {R.SEC_DIR}/nb11.py
# # NB11 - Statistics, tables and paper figures (CPU only)
# Reads **only** `preds_*.csv` files (plan 8), so no GPU is needed.  Protocol (plan 7):
# * every number per child; threshold fixed on out-of-fold validation predictions at 85 %
#   sensitivity, then applied unchanged to the locked test set
# * 95 % bootstrap CIs (2,000 resamples by child); DeLong (AUROC) and McNemar (paired errors)
#   vs the proposed model, Holm-corrected
# * calibration (ECE, Brier, reliability), decision curves vs **M-CHAT-R/F alone** and
#   **app + M-CHAT**, triage (refer rate + accuracy on the rest), fairness by sex / age band /
#   device / site
# * TRIPOD+AI checklist skeleton written next to the tables
# 

import numpy as np, pandas as pd
from pathlib import Path
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
OUT = U.WORK / "nb11"; OUT.mkdir(parents=True, exist_ok=True)
man = pd.read_csv(U.find_path("manifest.csv"))
SYN = bool(man.is_synthetic.iloc[0])
N_BOOT = 300 if PREFLIGHT else 2000
TARGET_SENS = 0.85
PROPOSED = "fusion_xmf_noquest"   # headline model: no questionnaire input (avoids overlap with the reference)
preds = U.load_all_preds([U.INPUT_ROOT, U.WORK])
preds = preds[preds.child_id.isin(man.child_id)]
print("prediction files:", preds.src_file.nunique(), "| models:", preds.model.nunique())


# ## 1. M-CHAT-R/F alone and app + M-CHAT (the clinical bar, plan 5.3)
# 

from sklearn.linear_model import LogisticRegression
if "mchat_total" in man and man.mchat_total.notna().any():
    rows = []
    for s in sorted(preds.seed.dropna().unique().astype(int)):
        for f in sorted(man[f"fold_s{s}"][man[f"fold_s{s}"] >= 0].unique()):
            tr, va, te = U.fold_ids(man, s, f)
            for part, ids in (("val", va), ("test", te)):
                d = man.set_index("child_id").loc[ids]
                rows.append(pd.DataFrame({"child_id": ids, "y_true": d.label.values, "prob": d.mchat_total.values / 20.0,
                                          "part": part, "model": "MCHAT_alone", "seed": s, "fold": f}))
    mchat = pd.concat(rows)
    combo = []
    app = preds[preds.model == PROPOSED]
    for (s, f), g in app.groupby(["seed", "fold"]):
        tr, va, te = U.fold_ids(man, int(s), int(f))
        oof_tr = app[(app.seed == s) & (app.part == "val") & app.child_id.isin(tr)]
        oof_tr = oof_tr.groupby("child_id").prob.mean()
        if len(oof_tr) < 10:
            continue
        Xtr = np.c_[U.logit(oof_tr.values), man.set_index("child_id").loc[oof_tr.index, "mchat_total"].values]
        lr = LogisticRegression().fit(Xtr, man.set_index("child_id").loc[oof_tr.index, "label"])
        for part in ("val", "test"):
            q = g[g.part == part]
            Xq = np.c_[U.logit(q.prob.values), man.set_index("child_id").loc[q.child_id, "mchat_total"].values]
            combo.append(q.assign(prob=lr.predict_proba(Xq)[:, 1], model="APP_plus_MCHAT"))
    preds = pd.concat([preds, mchat] + combo, ignore_index=True)


# ## 2. Main table: OOF and locked-test metrics with 95 % CIs
# 

summ, store = [], {}
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, TARGET_SENS, N_BOOT)
    store[m] = s
    for part in ("oof", "test"):
        if f"{part}_ci" not in s:
            continue
        ci = s[f"{part}_ci"].set_index("metric")
        summ.append({"model": m, "part": part, "n": s[part]["n"], "thr": round(s["thr"], 3),
                     **{k: f"{ci.loc[k,'est']:.3f} [{ci.loc[k,'lo']:.3f}, {ci.loc[k,'hi']:.3f}]" for k in ci.index}})
summ = pd.DataFrame(summ)
summ.to_csv(OUT / "table_main_metrics_ci.csv", index=False)
print(U.table_md(summ[summ.part == "test"][["model", "n", "auroc", "auprc", "sens", "spec", "ppv", "npv", "bacc", "ece"]]))


# ## 3. Paired tests vs the proposed model (locked test set), Holm-corrected
# 

tests = []
if PROPOSED in store and "test_df" in store[PROPOSED]:
    ref = store[PROPOSED]["test_df"].set_index("child_id")
    thr_ref = store[PROPOSED]["thr"]
    for m, s in store.items():
        if m == PROPOSED or "test_df" not in s:
            continue
        j = ref.join(s["test_df"].set_index("child_id"), rsuffix="_o", how="inner")
        if len(j) < 10:
            continue
        dl = U.delong_test(j.y_true, j.prob, j.prob_o)
        mc = U.mcnemar_test(j.y_true, j.prob >= thr_ref, j.prob_o >= s["thr"])
        tests.append({"comparator": m, "n": len(j), "auc_proposed": dl["auc1"], "auc_comparator": dl["auc2"],
                      "delong_p": dl["p"], "mcnemar_p": mc["p"]})
tests = pd.DataFrame(tests)
if len(tests):
    tests["delong_p_holm"] = U.holm(tests.delong_p.values)
    tests["mcnemar_p_holm"] = U.holm(tests.mcnemar_p.values)
    print(U.table_md(tests.sort_values("auc_comparator", ascending=False)))
tests.to_csv(OUT / "table_paired_tests.csv", index=False)


# ## 4. Calibration, decision curves, ROC (figures at 300 dpi)
# 

show = [m for m in (PROPOSED, "fusion_xmf", "fusion_late_avg_noquest", "fusion_early_concat_noquest", "frame_efficientnet_b0_expr",
                    "video_temporal_transformer", "MCHAT_alone", "APP_plus_MCHAT") if m in store and "test_df" in store[m]]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for m in show:
    t = store[m]["test_df"]
    fpr, tpr, _ = roc_curve(t.y_true, t.prob)
    ax[0].plot(fpr, tpr, label=f"{m} ({U._safe_auc(t.y_true, t.prob):.2f})")
    rc = U.reliability_curve(t.y_true, t.prob, 8); ax[1].plot(rc.p_mean, rc.y_mean, "o-", ms=3, label=m)
    dc = U.decision_curve(t.y_true, t.prob); ax[2].plot(dc.thr, dc.nb_model, label=m)
ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set(title="ROC - locked test set", xlabel="1 - specificity", ylabel="sensitivity")
ax[1].plot([0, 1], [0, 1], "k:"); ax[1].set(title="Calibration", xlabel="predicted risk", ylabel="observed ASD rate")
if show:
    ax[2].plot(dc.thr, dc.nb_all, "k--", label="refer all"); ax[2].axhline(0, color="k", lw=0.6)
ax[2].set(title="Decision curve", xlabel="threshold probability", ylabel="net benefit")
for a in ax: a.legend(fontsize=6)
if SYN: fig.suptitle("SYNTHETIC DEV COHORT - NOT RESULTS", color="red")
plt.tight_layout(); plt.savefig(OUT / "fig_roc_calibration_dca.png", dpi=300); plt.show()


# ## 5. Triage: refer the most uncertain children (evidential uncertainty)
# 

tri = []
if PROPOSED in store and "test_df" in store[PROPOSED]:
    t = store[PROPOSED]["test_df"].dropna(subset=["uncertainty"])
    for q in (0.0, 0.05, 0.10, 0.15, 0.20, 0.30):
        tri.append({"refer_quantile": q, **U.triage_metrics(t.y_true, t.prob, t.uncertainty, store[PROPOSED]["thr"], q)})
tri = pd.DataFrame(tri)
print(U.table_md(tri[[c for c in tri.columns if c in ("refer_quantile", "refer_rate", "kept_n", "kept_auroc", "kept_sens", "kept_spec", "kept_bacc")]]) if len(tri) else "no uncertainty")
tri.to_csv(OUT / "table_triage.csv", index=False)


# ## 6. Fairness (sex, age band, device, site) for the proposed model
# 

if PROPOSED in store and "test_df" in store[PROPOSED]:
    t = store[PROPOSED]["test_df"].merge(man[["child_id", "sex", "age_band", "device", "site"]], on="child_id")
    fair = U.fairness_table(t, ["sex", "age_band", "device", "site"], store[PROPOSED]["thr"])
    print(U.table_md(fair)); fair.to_csv(OUT / "table_fairness.csv", index=False)


# ## 7. Ablation / shortcut / robustness table and TRIPOD+AI skeleton
# 

abl = summ[(summ.part == "oof") & summ.model.str.startswith(("ablation_", "SHORTCUT_", "fusion_xmf"))][["model", "auroc", "bacc", "ece"]]
print(U.table_md(abl)); abl.to_csv(OUT / "table_ablations_shortcuts.csv", index=False)
tripod = """# TRIPOD+AI checklist - pointers (fill page numbers in the manuscript)
1 Title: screening (not diagnosis) model, children 18-72 months, Bangladesh
3 Background / 4 Objectives: screening vs M-CHAT-R/F; decision support only
5 Data sources: prospective protocol recordings, sites, dates  (NB00 audit_report.json)
6 Participants: eligibility, ASD / TD / DD groups, consent
7 Data preparation: NB01 / NB02 pipelines, missing-modality handling
8 Outcome: clinician DSM-5 (+ADOS-2) diagnosis blind to the model
9 Predictors: face video, voice, cry, M-CHAT items (reported with/without)
10 Sample size: statistician calculation (attach)
11 Missing data: learned missing tokens + modality dropout
12 Analysis: child-wise CV, locked test (NB00), threshold at 85 % sensitivity on OOF
13 Class imbalance / 14 Fairness: table_fairness.csv
15 Model output: risk + uncertainty, 3-way decision (table_triage.csv)
16 Training vs evaluation: table_main_metrics_ci.csv (OOF vs locked test)
20 Performance: AUROC/AUPRC/sens/spec/PPV/NPV/calibration with 95 % CIs
22 Model updating / 23 interpretability: NB10 outputs
25 Shortcut / confounding analysis: table_ablations_shortcuts.csv
27 Code & data availability: Kaggle notebooks + asd_lib; data per consent
"""
(OUT / "TRIPOD_AI_checklist_skeleton.md").write_text(tripod)
if SYN:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb11')

In [ ]:
R.finish()